In [ ]:
# ── Starter Cell 1: Load environment variables ──────────────────────────────
from dotenv import load_dotenv   # imports load_dotenv to read the .env file
import os                        # imports os so we can call os.getenv()

load_dotenv()                    # reads .env in the current directory and sets env vars
api_key = os.getenv("ANTHROPIC_API_KEY")  # retrieves the key; returns None if missing

print("API key loaded:", api_key is not None)  # prints True when the key was found

In [ ]:
# ── Starter Cell 2: Initialise the Anthropic client ─────────────────────────
from anthropic import Anthropic   # imports the Anthropic SDK client class

client = Anthropic(api_key=api_key)  # creates the client; api_key authenticates every request
print("Anthropic client ready")       # confirms the object was constructed without error

In [ ]:
# ── Starter Cell 3: Smoke test — verifies end-to-end connectivity ────────────
response = client.messages.create(   # calls the Messages API
    model="claude-haiku-4-5",        # fastest/cheapest model — ideal for smoke tests
    #    model="claude-sonnet-4-5",  # uncomment for higher quality during development
    max_tokens=50,                   # short cap — we only need one sentence back
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]  # minimal prompt
)
print(response.content[0].text)  # content is a list; [0] is the first (only) block; .text is the string

# CCA Lab: Temperature — Controlling Creativity and Determinism

**Course:** Building with the Claude API  
**Sub-module:** Accessing Claude with the API  
**Lesson:** Temperature

## What this lab covers
- How Claude generates text (tokenisation → prediction → sampling)
- The `temperature` parameter: range, effect, and best-practice values
- Running the same prompt at temperatures 0.0, 0.5, and 1.0 and comparing outputs
- Matching temperature ranges to task categories
- Improvement cycle: writing, scoring, and iterating prompts with a numeric rubric
- Failure mode analysis with live code demonstrations
- Token-usage tracking across every API call in the session

## CCA domains exercised
- **Primary:** Prompt Engineering  
- **Contributing:** Agentic Architecture (API parameter reference)

## Learning objectives
1. Explain what temperature does to token-selection probabilities.
2. Choose an appropriate temperature range for a given task category.
3. Implement and test a `chat()` helper that accepts a `temperature` argument.
4. Apply a numeric rubric to score and iteratively improve prompts.
5. Identify and recover from common temperature-related failure modes.

## Section 1: API Parameter Reference
## CCA objective demonstrated: Agentic Architecture — API Parameter Reference: Temperature and Related Sampling Controls

Before writing any code, know every knob you are turning. The table below covers all
parameters used in this lab.

| Parameter | Type | Required | Default | Purpose |
|-----------|------|----------|---------|-----------------------------------------------------------|
| `model` | string | ✅ Yes | — | Selects which Claude model handles the request |
| `max_tokens` | integer | ✅ Yes | — | Hard cap on output length; prevents runaway responses |
| `messages` | list[dict] | ✅ Yes | — | Ordered conversation history (`role` + `content` pairs) |
| `temperature` | float | ❌ No | `1.0` | Controls randomness: 0 = deterministic, 1 = most varied |
| `system` | string | ❌ No | `None` | Persistent persona / constraint injected before the conversation |

> **Architect insight:** `temperature` and `max_tokens` are the two most impactful
> parameters for controlling output quality in production. Set `max_tokens` first
> (safety), then tune `temperature` for the task type.

## Section 2: ask() Helper with Statement-by-Statement Annotation
## CCA objective demonstrated: Prompt Engineering — Implementing a reusable, temperature-aware chat helper

In [ ]:
# ── ask() / chat() helper — full statement-by-statement annotation ───────────

usage_log = []   # session-wide list; every API call appends one dict here for token tracking

def chat(messages, system=None, temperature=1.0, label="call"):
    """
    Send a conversation to Claude and return the assistant's reply as a string.

    Parameters
    ----------
    messages    : list[dict]  — conversation history, each dict has 'role' and 'content'
    system      : str | None  — optional system prompt injected before the conversation
    temperature : float       — sampling temperature in [0.0, 1.0]; default 1.0 (API default)
    label       : str         — human-readable tag stored in usage_log for this call

    Returns
    -------
    str — the plain-text content of Claude's first response block
    """

    params = {                        # build the API payload as a dict so we can conditionally add keys
        "model": "claude-haiku-4-5", # model name; swap to claude-sonnet-4-5 for higher quality
        "max_tokens": 1000,          # output cap; prevents accidental oversized responses
        "messages": messages,        # the full conversation list — Claude needs full history for context
        "temperature": temperature,  # float in [0,1]; lower = more deterministic, higher = more creative
    }

    if system:                        # only add 'system' key when a value was supplied
        params["system"] = system    # system is a top-level key, NOT inside messages[]
                                     # Why? The API treats system as a separate instruction channel
                                     # that persists across the whole conversation

    message = client.messages.create(**params)  # unpacks params dict as keyword args; makes the HTTP call
    # message.content is a LIST because Claude can return multiple content blocks
    # (e.g. text block + tool_use block).  We always want the first text block → [0].text

    reply = message.content[0].text  # [0] = first block; .text = the string payload

    # ── Token tracking ───────────────────────────────────────────────────────
    usage_log.append({               # record usage metadata for this call
        "label": label,              # human tag so we know which call this was
        "temperature": temperature,  # log the temperature so comparisons are meaningful
        "input_tokens": message.usage.input_tokens,   # tokens consumed by the prompt
        "output_tokens": message.usage.output_tokens, # tokens produced in the reply
        "stop_reason": message.stop_reason,           # 'end_turn' = natural finish; 'max_tokens' = truncated
    })
    # stop_reason matters: if it is 'max_tokens' the response was CUT OFF — increase max_tokens

    return reply  # return the plain-text string to the caller

## Section 3: Temperature and Sampling — Controlling Creativity and Determinism
## CCA objective demonstrated: Prompt Engineering — Comparison of temperature 0.0 / 0.5 / 1.0 with task-category mapping

### How Claude generates text
1. **Tokenisation** — your prompt is split into sub-word tokens.
2. **Prediction** — the model assigns a probability to every possible next token.
3. **Sampling** — one token is drawn according to those probabilities; the cycle repeats.

`temperature` reshapes the probability distribution *before* sampling:
- **0.0** → argmax: always pick the single highest-probability token (fully deterministic).
- **1.0** → raw distribution: probabilities are used as-is (most varied).
- Values in between interpolate smoothly.

### Task-category → temperature mapping

| Temperature range | Task category | Examples |
|-------------------|---------------|----------|
| 0.0 – 0.3 | Factual / analytical | Coding, data extraction, content moderation |
| 0.4 – 0.7 | Balanced | Summarisation, educational content, constrained creative writing |
| 0.8 – 1.0 | Exploratory / creative | Brainstorming, marketing copy, jokes, open-ended ideation |

> **Architectural principle:** Temperature does not *guarantee* variety — it adjusts
> *probability*. Even at 1.0 Claude may repeat itself; even at 0.0 slight non-determinism
> can occur across hardware configurations.

In [ ]:
# ── Temperature comparison: same prompt, 3 runs at 0.0 / 0.5 / 1.0 ──────────
# 3 runs per temperature makes variance VISIBLE — 0.0 is consistent, 1.0 varies.

prompt = "Give me a one-sentence movie idea."
N_RUNS = 3

for temp in [0.0, 0.5, 1.0]:
    print(f"Temperature {temp} — {N_RUNS} runs:")
    print("-" * 70)
    for run in range(1, N_RUNS + 1):
        msgs = [{"role": "user", "content": prompt}]
        reply = chat(msgs, temperature=temp, label=f"temp_{temp}_r{run}")
        truncated = reply[:90].replace(chr(10), " ")
        print(f"  Run {run}: {truncated}")
    print()

print("Observation: temperature 0.0 runs are nearly identical.")
print("             temperature 1.0 runs vary noticeably.")
print("             temperature 0.5 is somewhere in between.")

## Section 4: System Prompt — What It Is, When to Use It, and Why It Matters
## CCA objective demonstrated: Prompt Engineering — System parameter design and decision framework

The `system` parameter is a special instruction channel that sits *above* the conversation.
Claude reads it before any user message, and it persists for the entire session.

### System vs. user-turn — decision table

| Belongs in `system` | Belongs in the user turn |
|---------------------|--------------------------|
| Persona / role definition | The actual task or question |
| Output format requirements | Dynamic data (user name, date, inputs) |
| Hard constraints (never do X) | Follow-up clarifications |
| Domain restrictions (only discuss Y) | Examples specific to this request |
| Tone and style guidelines | Conversational context |

**Architectural principle:** Put anything that must be *constant across every turn* in
the system parameter; put anything *specific to this request* in the user turn.

In [ ]:
# ── System prompt demo: same persona, 3 runs at low vs high temperature ──────
# System prompt constrains FORMAT; temperature controls word-choice variance.

creative_system = (
    "You are a Hollywood pitch artist. "
    "Always respond with exactly ONE sentence. "
    "Be vivid and dramatic."
)

task = "Give me a movie idea."

for temp, label in [(0.1, "LOW  (0.1)"), (0.9, "HIGH (0.9)")]:
    print(f"── Temperature {label} — 3 runs ──────────────────────────────────")
    for run in range(1, 4):
        msgs = [{"role": "user", "content": task}]
        reply = chat(msgs, system=creative_system, temperature=temp,
                     label=f"sys_{int(temp*10)}_r{run}")
        print(f"  Run {run}: {reply[:100].replace(chr(10), ' ')}")
    print()

print("Observation: every run obeys ONE-sentence constraint (system prompt works).")
print("             LOW temp runs are more similar; HIGH temp runs vary more.")

## Section 5: Rubric Scoring — Operationalising Quality
## CCA objective demonstrated: Prompt Engineering — Numeric rubric design and per-criterion printed output

In [ ]:
# ── Rubric scoring function ───────────────────────────────────────────────────
import re   # standard library regex module — used for pattern matching in responses

def score_movie_pitch(text):
    """
    Score a one-sentence movie pitch against four quality dimensions.

    Each dimension is worth 1–3 points. Maximum total = 10 points.
    Prints each dimension with a pass/fail icon and its numeric contribution.

    Parameters
    ----------
    text : str — Claude's raw response string

    Returns
    -------
    int — total score (0–10)
    """
    total = 0   # accumulator — int, not float, because all scores are whole numbers

    # ── Dimension 1: Length (max 2 pts) ──────────────────────────────────────
    word_count = len(text.split())          # .split() splits on any whitespace; len() counts tokens
    if word_count <= 30:                    # one sentence should be ≤ 30 words
        d1 = 2                             # full marks — concise
    elif word_count <= 50:                 # slightly over but still readable
        d1 = 1
    else:
        d1 = 0                             # too long for a pitch
    icon1 = "✅" if d1 > 0 else "❌"       # bool → emoji for at-a-glance reading
    print(f"{icon1} Dimension 1 — Conciseness ({word_count} words): {d1}/2 pts")
    total += d1                            # int addition; += is clearer than total = total + d1

    # ── Dimension 2: Character present (max 2 pts) ───────────────────────────
    # re.search scans the WHOLE string (not just the start, unlike re.match)
    # \b word boundaries prevent 'archaeologist' matching inside 'archaeologists'
    character_words = r"\b(hero|villain|detective|scientist|warrior|archaeologist|agent|doctor|pilot|spy)\b"
    has_character = bool(re.search(character_words, text, re.IGNORECASE))  # bool() converts Match→True, None→False
    d2 = 2 if has_character else 0         # int(bool()) pattern: True→1, but we want 2 pts so we use if/else
    icon2 = "✅" if d2 > 0 else "❌"
    print(f"{icon2} Dimension 2 — Named character type: {d2}/2 pts")
    total += d2

    # ── Dimension 3: Conflict signal (max 3 pts) ─────────────────────────────
    conflict_words = r"\b(must|against|threat|danger|secret|escape|save|prevent|stop|betrayal|mystery|war)\b"
    conflict_matches = re.findall(conflict_words, text, re.IGNORECASE)  # returns LIST of all matches
    d3 = min(len(conflict_matches), 3)     # cap at 3; min() avoids over-scoring verbose responses
    icon3 = "✅" if d3 > 0 else "❌"
    print(f"{icon3} Dimension 3 — Conflict signal ({len(conflict_matches)} match(es)): {d3}/3 pts")
    total += d3

    # ── Dimension 4: Setting detail (max 3 pts) ──────────────────────────────
    setting_words = r"\b(space|ocean|city|ancient|future|island|forest|underground|desert|mountain|dystopian)\b"
    setting_matches = re.findall(setting_words, text, re.IGNORECASE)
    d4 = min(len(setting_matches), 3)      # same cap logic as d3
    icon4 = "✅" if d4 > 0 else "❌"
    print(f"{icon4} Dimension 4 — Setting detail ({len(setting_matches)} match(es)): {d4}/3 pts")
    total += d4

    print(f"   ── Total score: {total}/10")
    return total   # return int so callers can compare scores numerically


# ── Quick smoke test of the rubric ───────────────────────────────────────────
sample = "A brave detective must stop an ancient secret society from destroying the future city."
print("Sample pitch:", sample)
print()
_ = score_movie_pitch(sample)   # _ discards the return value — we only want the printed output here

## Section 6: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
## CCA objective demonstrated: Prompt Engineering — Iterative prompt refinement with numeric rubric

The improvement cycle has four steps:
1. **Write** a baseline prompt.
2. **Evaluate** it with the rubric.
3. **Improve** the prompt based on which dimensions scored lowest.
4. **Re-evaluate** and confirm the score rose.

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs.

In [ ]:
# ── Improvement cycle ─────────────────────────────────────────────────────────

PASS_THRESHOLD = 7   # scores >= 7/10 are considered production-ready

# ── Version 1: Baseline — vague prompt, default temperature ──────────────────
v1_prompt = "Give me a movie idea."                  # intentionally weak — no constraints
v1_msgs   = [{"role": "user", "content": v1_prompt}]
v1_reply  = chat(v1_msgs, temperature=1.0, label="cycle_v1")   # high temp for creativity
print("V1 reply:", v1_reply[:120])
print()
v1_score  = score_movie_pitch(v1_reply)              # evaluate against rubric
print()

# ── Version 2: Improved — explicit constraints added ─────────────────────────
v2_prompt = (
    "Give me a one-sentence movie idea that features a named character type (like detective "
    "or scientist), a clear conflict they must overcome, and a vivid setting."
)  # added: length guidance, character requirement, conflict word, setting requirement
v2_msgs   = [{"role": "user", "content": v2_prompt}]
v2_reply  = chat(v2_msgs, temperature=0.7, label="cycle_v2")   # medium temp — creative but focused
print("V2 reply:", v2_reply[:120])
print()
v2_score  = score_movie_pitch(v2_reply)
print()

# ── Version 3: Further refined — system prompt + tighter temperature ──────────
v3_system = "You are a Hollywood pitch artist. Always respond with exactly one sentence."
v3_prompt = (
    "Pitch a movie set in an ancient or futuristic location where a hero or detective "
    "must stop a secret threat that endangers the city."
)  # vocabulary directly targets rubric dimensions
v3_msgs   = [{"role": "user", "content": v3_prompt}]
v3_reply  = chat(v3_msgs, system=v3_system, temperature=0.5, label="cycle_v3")
print("V3 reply:", v3_reply[:120])
print()
v3_score  = score_movie_pitch(v3_reply)
print()

# ── Side-by-side comparison table ────────────────────────────────────────────
print("=" * 72)
print(f"{'Version':<10} {'Temp':<6} {'Score':<8} {'Prompt (truncated)'}")
print("-" * 72)

# Build rows as a list first — avoids backslash-in-f-string SyntaxError in Python 3.11
rows = [
    ("V1", "1.0",  v1_score, v1_prompt[:45]),
    ("V2", "0.7",  v2_score, v2_prompt[:45]),
    ("V3", "0.5",  v3_score, v3_prompt[:45]),
]
for ver, tmp, score, trunc in rows:                      # unpack each tuple
    print(f"{ver:<10} {tmp:<6} {score:<8} {trunc}...")

print("=" * 72)
best_score = max(v1_score, v2_score, v3_score)           # max() over three ints
result_label = "PASS ✅" if best_score >= PASS_THRESHOLD else "FAIL ❌"  # conditional label
print(f"Best score: {best_score}/10 — {result_label} (threshold = {PASS_THRESHOLD})")

## Section 7: Failure Mode Analysis
## CCA objective demonstrated: Prompt Engineering — Identifying, cataloguing, and recovering from temperature-related failures

| Failure Mode | Trigger | Symptom |
|---|---|---|
| **Out-of-range temperature** | Passing `temperature` < 0 or > 1 | API raises `BadRequestError` with a validation message |
| **Wrong type for temperature** | Passing a string (`"high"`) instead of a float | `TypeError` before the request is even sent |
| **max_tokens too low** | Setting `max_tokens=1` for a long task | `stop_reason` is `max_tokens`; response is truncated mid-sentence |
| **Temp 0.0 with open-ended creative task** | Asking for variety at temperature 0 | Every run returns an identical (often generic) answer |
| **Temp 1.0 for structured extraction** | Using high temperature for JSON/code output | Output varies between runs; schema breaks downstream parsers |

In [ ]:
# ── Failure mode live demo ────────────────────────────────────────────────────
import anthropic   # import for the exception class

# ── Demo 1: temperature out of range (> 1.0) ─────────────────────────────────
print("Demo 1: temperature = 1.5 (out of range)")
try:
    bad_reply = client.messages.create(   # make a direct API call (bypassing our helper)
        model="claude-haiku-4-5",
        max_tokens=50,
        temperature=1.5,                  # ← deliberately invalid: valid range is 0.0–1.0
        messages=[{"role": "user", "content": "Hello"}]
    )
except anthropic.BadRequestError as e:    # SDK raises BadRequestError for 400-level API errors
    print(f"  ✅ Caught BadRequestError: {str(e)[:120]}")   # [:120] prevents flooding the output
except Exception as e:                    # catch any other unexpected error type
    print(f"  ⚠️  Unexpected error type {type(e).__name__}: {e}")

print()

# ── Demo 2: max_tokens truncation (stop_reason = 'max_tokens') ────────────────
print("Demo 2: max_tokens=5 forces truncation")
truncated_msg = client.messages.create(   # direct call so we can inspect .stop_reason
    model="claude-haiku-4-5",
    max_tokens=5,                         # ← far too small for any useful response
    temperature=0.0,
    messages=[{"role": "user", "content": "Describe the ocean in three sentences."}]
)
stop = truncated_msg.stop_reason          # 'max_tokens' confirms truncation occurred
text = truncated_msg.content[0].text      # the truncated fragment
print(f"  stop_reason : {stop}")
print(f"  reply fragment: '{text}'")
if stop == "max_tokens":                  # programmatic check — production code should do this
    print("  ❌ Response was truncated — increase max_tokens or shorten the task.")

# Log these calls for token tracking (manual append since we bypassed chat())
usage_log.append({
    "label": "failure_demo_truncation",
    "temperature": 0.0,
    "input_tokens": truncated_msg.usage.input_tokens,
    "output_tokens": truncated_msg.usage.output_tokens,
    "stop_reason": truncated_msg.stop_reason,
})

## Section 8: Token Usage Tracking
## CCA objective demonstrated: Agentic Architecture — Monitoring per-call and cumulative token consumption

In [ ]:
# ── Token usage report — per-call table with cumulative totals ────────────────

print(f"{'#':<4} {'Label':<28} {'Temp':<6} {'Input':>7} {'Output':>8} {'Cum In':>9} {'Cum Out':>9} {'Stop'}")
print("-" * 88)   # separator wide enough for all columns

cum_in  = 0   # running total of input tokens across the session
cum_out = 0   # running total of output tokens across the session

for i, entry in enumerate(usage_log, start=1):   # enumerate so we get a 1-based row number
    cum_in  += entry["input_tokens"]              # accumulate input
    cum_out += entry["output_tokens"]             # accumulate output

    # Build display strings outside the f-string to avoid backslash-in-f-string issues
    label   = entry["label"][:26]                 # truncate long labels to keep columns aligned
    temp    = str(entry["temperature"])           # convert float to string for formatting
    inp     = entry["input_tokens"]
    out     = entry["output_tokens"]
    stop    = entry["stop_reason"]

    print(f"{i:<4} {label:<28} {temp:<6} {inp:>7} {out:>8} {cum_in:>9} {cum_out:>9} {stop}")

print("=" * 88)
total_tokens = cum_in + cum_out   # grand total for cost estimation
print(f"{'TOTAL':<4} {'':28} {'':6} {cum_in:>7} {cum_out:>8} {'':>9} {'':>9}")
print(f"Session grand total: {total_tokens:,} tokens")
print()
# Haiku pricing reference (as of 2025): input ~$0.80/M, output ~$4.00/M
est_cost = (cum_in * 0.80 + cum_out * 4.00) / 1_000_000   # rough USD estimate
print(f"Estimated cost (Haiku pricing): ${est_cost:.5f} USD")

## Section 9: Multi-turn Conversation — Messages List Accumulation
## CCA objective demonstrated: Prompt Engineering — Building stateful conversation by accumulating the messages list

In [ ]:
# ── Multi-turn conversation demo: temperature's role in ongoing dialogue ───────

conversation = []   # start with an empty messages list — each turn appends to this

# ── Turn 1 ────────────────────────────────────────────────────────────────────
user_msg_1 = "What is temperature in language models? Answer in two sentences."
conversation.append({"role": "user", "content": user_msg_1})  # add the user turn

reply_1 = chat(conversation, temperature=0.2, label="multi_turn_1")  # low temp → precise definition
conversation.append({"role": "assistant", "content": reply_1})       # add the assistant reply

print("After Turn 1 — full messages list:")
for m in conversation:    # print every entry so the list accumulation is visible
    role    = m["role"]
    snippet = m["content"][:100].replace("\n", " ")
    print(f"  [{role}]: {snippet}")
print()

# ── Turn 2 ────────────────────────────────────────────────────────────────────
user_msg_2 = "Now give me a creative analogy that explains it to a child."
conversation.append({"role": "user", "content": user_msg_2})  # append follow-up

reply_2 = chat(conversation, temperature=0.9, label="multi_turn_2")  # high temp → imaginative analogy
conversation.append({"role": "assistant", "content": reply_2})

print("After Turn 2 — full messages list:")
for m in conversation:
    role    = m["role"]
    snippet = m["content"][:100].replace("\n", " ")
    print(f"  [{role}]: {snippet}")
print()

# ── Turn 3 ────────────────────────────────────────────────────────────────────
user_msg_3 = "Summarise both answers in one sentence."
conversation.append({"role": "user", "content": user_msg_3})

reply_3 = chat(conversation, temperature=0.0, label="multi_turn_3")  # temp 0 → deterministic summary
conversation.append({"role": "assistant", "content": reply_3})

print("After Turn 3 — full messages list (6 entries):")
for idx, m in enumerate(conversation, 1):   # enumerate so we can see the list grow
    role    = m["role"]
    snippet = m["content"][:100].replace("\n", " ")
    print(f"  {idx}. [{role}]: {snippet}")
print()
print("Key insight: Claude sees the ENTIRE list on every call — that's how it")
print("maintains context across turns. Temperature can differ turn-by-turn.")

## Section 10: Practice Drills
## CCA objective demonstrated: Prompt Engineering — Independent application of temperature control patterns

Complete the three exercises below by filling in the `# YOUR CODE HERE` sections.

In [ ]:
# ── Drill 1: Choose the right temperature for a deterministic task ────────────
# Task: extract name and DOB — fixed answer, not creative.
# Observe which temperature gives the most consistent output across runs.

print("Drill 1 — extraction at 3 temperatures:")
for t in [0.0, 0.5, 1.0]:
    msgs = [{"role": "user", "content":
        "Extract name and DOB from: Alice Smith was born on 1990-05-14. "
        "Reply with only: Name: X, DOB: Y"}]
    reply = chat(msgs, temperature=t, label=f"drill1_t{t}")
    print(f"  temp={t}: {reply[:80].replace(chr(10), ' ')}")
print("Which temperature gives the most consistent result? (answer: near 0.0)")

print()

# ── Drill 2: Use temperature to maximise creative variety ─────────────────────
# Task: generate marketing taglines — creative task, higher temp = more variety.
# Run 3 times at each temperature and observe consistency vs variance.

print("Drill 2 — tagline variety at 3 temperatures (3 runs each):")
for temp in [0.0, 0.5, 1.0]:
    print(f"  Temperature {temp}:")
    for run in range(1, 4):
        msgs = [{"role": "user", "content":
            "Write one punchy 5-word marketing tagline for a coffee brand."}]
        reply = chat(msgs, temperature=temp, label=f"drill2_t{temp}_r{run}")
        print(f"    Run {run}: {reply[:60].replace(chr(10), ' ')}")
print("Which temperature produces the most variety? (answer: 1.0)")

print()

# ── Drill 3: Improvement cycle — score >= 8/10 on the movie pitch rubric ──────
# The rubric checks: character word, conflict word, setting word, conciseness.
# Edit my_prompt below until score_movie_pitch() returns >= 8.

my_prompt = (
    "Give me a one-sentence movie idea where a detective must stop a secret "
    "ancient danger threatening a futuristic ocean city."
)   # edit this until you score >= 8/10
# Hint: the rubric rewards: character word (detective/hero/spy...),
#       conflict word (must/stop/threat/secret/save/prevent...),
#       setting word (ancient/future/ocean/city/space/desert...),
#       and conciseness (under 30 words = full marks).

my_msgs  = [{"role": "user", "content": my_prompt}]
my_reply = chat(my_msgs, temperature=0.3, label="drill3")
print("Drill 3 reply:", my_reply[:120])
print()
my_score = score_movie_pitch(my_reply)
print("Result:", "PASS" if my_score >= 8 else "FAIL — edit my_prompt and re-run")

## Section 11: CCA Takeaways — 5 Exam-Ready Mental Models
## CCA objective demonstrated: All domains — consolidation of lab concepts into transferable principles

| # | Mental Model | Exam cue |
|---|---|---|
| 1 | **Temperature reshapes probabilities, not content** | Even at 1.0 Claude may repeat itself; at 0.0 slight variation can still occur across deployments. |
| 2 | **Match temperature to task entropy** | Deterministic tasks (extraction, code) → low temp; open-ended tasks (brainstorming, copy) → high temp. |
| 3 | **System prompt = constant channel; user turn = dynamic channel** | Anything that must hold across every turn belongs in `system`, not repeated in each message. |
| 4 | **stop_reason is the safety net** | Always check `stop_reason`; `max_tokens` means the response was cut off — raise `max_tokens` or redesign the task. |
| 5 | **Rubric + judge layering beats keyword-only grading** | Keyword rubrics are fast and deterministic; Claude-as-judge catches semantic errors keywords miss. Use both. |